### Problem 001: Network Delay Time (LeetCode 743)

### Problem Definition and Constraints
You are given a network of `n` directed nodes, labeled from `1` to `n`. 
You are given `times`, a list of directed edges where `times[i] = (u, v, t)`:
* `u` is the source node.
* `v` is the target node.
* `t` is the time it takes for a signal to travel from `u` to `v`.

You start by sending a signal from node `k`. 
Return the minimum time it takes for *all* `n` nodes to receive the signal. If it is impossible for all the nodes to receive the signal (e.g., someone is disconnected), return `-1`.

**Examples:**
* **Example 1:**
  * **Input:** `times = [[1,2,1],[2,3,1],[1,4,4],[3,4,1]], n = 4, k = 1`
  * **Output:** `3`
* **Example 2:**
  * **Input:** `times = [[1,2,1],[2,3,1]], n = 3, k = 2`
  * **Output:** `-1` (Node 1 can never be reached).

* Constraints:
  * 1 <= k <= n <= 100
  * 1 <= times.length <= 1000

### Dijkstra's Algorithm (Min-Heap) Approach
If this were an unweighted graph, we would use a standard BFS Queue. But because edges have different times, a path with 3 short jumps might be faster than a path with 1 massive jump. A standard Queue just blindly processes "1 jump away, 2 jumps away," which fails here.

**The Solution:** We replace the standard Queue with a **Min-Heap (Priority Queue)**. 
A Min-Heap automatically sorts everything inside it so that the item with the *lowest total time* is always at the very front. 
1. We start at node `k` at time `0`.
2. We pull the fastest available route out of the Min-Heap.
3. If we haven't visited that node yet, we lock it in (add to `visited` set). The time it took to get here is officially the fastest possible time.
4. We look at its neighbors, add the current time to the neighbor's travel time, and throw them into the Min-Heap.
5. We repeat until the Heap is empty. If we visited `n` unique nodes, the time it took to reach the very last node is our answer.

* **Time Complexity:** $O(E \log V)$ — $E$ is the number of edges and $V$ is the number of vertices (nodes). In the worst case, we push every single edge into the Min-Heap. Pushing to a heap takes logarithmic time.
* **Space Complexity:** $O(V + E)$ — To store the Adjacency List (which holds every node and edge) and the Min-Heap itself.

In [ ]:
import collections
import heapq
from typing import List

class Solution:
    def networkDelayTime(self, times: List[List[int]], n: int, k: int) -> int:
        # Step 1: Build the Adjacency List
        # Dictionary mapping: node -> list of (travel_time, neighbor_node)
        adj = collections.defaultdict(list)
        for u, v, t in times:
            adj[u].append((t, v))
            
        # Step 2: Initialize the Min-Heap and Visited Set
        # The heap stores tuples of (accumulated_time, node)
        # We start at node k with 0 accumulated time.
        min_heap = [(0, k)] 
        visited = set()
        
        # We will track the time it takes to reach the furthest node
        total_time = 0
        
        # Step 3: Dijkstra's Algorithm
        while min_heap:
            # heappop ALWAYS gives us the node with the lowest accumulated time
            time, node = heapq.heappop(min_heap)
            
            # THE BRAKES: If we already found a faster way to this node, skip it.
            if node in visited:
                continue
                
            # Lock in this node as visited
            visited.add(node)
            
            # Update our master clock to the time it took to reach this node
            total_time = max(total_time, time)
            
            # Step 4: Explore the neighbors
            for neighbor_time, neighbor in adj[node]:
                if neighbor not in visited:
                    # Accumulate the time: (time it took to get here + time to neighbor)
                    heapq.heappush(min_heap, (time + neighbor_time, neighbor))
                    
        # Step 5: Did the signal reach everyone?
        if len(visited) == n:
            return total_time
        else:
            return -1

### Problem 002: Min Cost to Connect Points (LeetCode 1584)

### Problem Definition and Constraints
You are given a 2-D integer array `points`, where `points[i] = [xi, yi]`. Each `points[i]` represents a distinct point on a 2-D plane.
The cost of connecting two points `[xi, yi]` and `[xj, yj]` is the **Manhattan distance** between the two points: `|xi - xj| + |yi - yj|`.

Return the minimum cost to connect all points together, such that there exists exactly one path between each pair of points. 

**Examples:**
* **Example 1:**
  * **Input:** `points = [[0,0],[2,2],[3,3],[2,4],[4,2]]`
  * **Output:** `10`

* Constraints:
  * 1 <= points.length <= 1000
  * -1,000,000 <= xi, yi <= 1,000,000
  * All pairs (xi, yi) are distinct.

### Minimum Spanning Tree (MST) & Kruskal's Algorithm
This problem is asking for a **Minimum Spanning Tree (MST)**. 
* **Spanning:** It must connect every single point together.
* **Tree:** There are no loops (cycles). 
* **Minimum:** It uses the absolute cheapest total cost to do it.

To solve this, we use **Kruskal's Algorithm** paired with a tool called **Union-Find (Disjoint Set Union - DSU)**. 

**The Strategy:**
1. **The Blueprint:** We calculate the cost (Manhattan distance) of building a bridge between *every single possible pair* of points on the board.
2. **The Sorting:** We take all those potential bridges and sort them from cheapest to most expensive.
3. **The Build:** We start buying the cheapest bridges one by one. 
4. **The Rule (Union-Find):** Before we build a bridge, we ask: *"Are these two points already connected by a different path?"* If they are, building this bridge would create a useless loop (a cycle), so we throw it away. If they aren't, we build it and add the cost to our total.
5. We stop when we have built exactly $n - 1$ bridges (because it takes exactly 4 bridges to connect 5 points).

**The Math & Complexity Breakdown:**
* **Time Complexity:** $O(n^2 \log n)$ 
  * Calculating every possible edge takes $O(n^2)$ time.
  * Sorting those $n^2$ edges takes $O(n^2 \log(n^2))$, which mathematically simplifies to $O(n^2 \log n)$. 
  * The Union-Find operations take nearly $O(1)$ time each, so sorting is the bottleneck.
* **Space Complexity:** $O(n^2)$ — We have to store every single possible edge in a list before we sort it. For $n$ points, there are roughly $n^2 / 2$ edges.

In [ ]:
from typing import List

# Helper Class: Union-Find (Disjoint Set)
class UnionFind:
    def __init__(self, n):
        # Initially, every point is its own boss (parent)
        self.parent = list(range(n))
        # Rank helps keep the tree flat when we merge groups
        self.rank = [1] * n

    def find(self, i):
        # Find the absolute top boss of the group
        if self.parent[i] != i:
            # Path compression: point directly to the top boss
            self.parent[i] = self.find(self.parent[i])
        return self.parent[i]

    def union(self, i, j):
        # Find the bosses of both points
        p1, p2 = self.find(i), self.find(j)
        
        # If they have the same boss, they are already connected! (Cycle detected)
        if p1 == p2:
            return False
            
        # If they have different bosses, merge them based on rank
        if self.rank[p1] > self.rank[p2]:
            self.parent[p2] = p1
        elif self.rank[p1] < self.rank[p2]:
            self.parent[p1] = p2
        else:
            self.parent[p2] = p1
            self.rank[p1] += 1
            
        return True

class Solution:
    def minCostConnectPoints(self, points: List[List[int]]) -> int:
        n = len(points)
        edges = []
        
        # Step 1: Generate all possible edges (The Blueprint)
        for i in range(n):
            for j in range(i + 1, n):
                x1, y1 = points[i]
                x2, y2 = points[j]
                # Manhattan distance formula
                dist = abs(x1 - x2) + abs(y1 - y2)
                
                # Store as (cost, point1_index, point2_index)
                edges.append((dist, i, j))
                
        # Step 2: Sort edges from cheapest to most expensive
        edges.sort()
        
        # Step 3: Kruskal's Algorithm using Union-Find
        uf = UnionFind(n)
        total_cost = 0
        edges_used = 0
        
        for dist, i, j in edges:
            # Try to connect point i and point j
            if uf.union(i, j): 
                # If True, they weren't connected yet. Build the bridge!
                total_cost += dist
                edges_used += 1
                
                # A tree connecting n points ALWAYS has exactly n - 1 edges
                if edges_used == n - 1:
                    break
                    
        return total_cost

### Problem 003: Cheapest Flights Within K Stops (LeetCode 787)

### Problem Definition and Constraints
You are given `n` airports, labeled `0` to `n - 1`. 
You are given a list of `flights` where `flights[i] = [source, destination, price]`. 
You want to fly from `src` to `dst`. 

Find the **cheapest** total price, but you are only allowed to make at most **`k` stops**. (Note: `k` stops means taking at most `k + 1` actual flights). If it is impossible, return `-1`.

**Examples:**
* **Example 1:**
  * **Input:** `n = 4, flights = [[0,1,200],[1,2,100],[1,3,300],[2,3,100]], src = 0, dst = 3, k = 1`
  * **Output:** `500`
  * *Explanation:* Path `0 -> 1 -> 2 -> 3` costs 400, but it has 2 stops. We are only allowed 1 stop. The path `0 -> 1 -> 3` costs 500 and only has 1 stop, so it is the winner.

* Constraints:
  * 1 <= n <= 100
  * 1 <= price <= 1000
  * 0 <= k < n

### The Bellman-Ford Algorithm (Layer-by-Layer)
Dijkstra's Algorithm finds the absolute cheapest path, but it doesn't care about the number of stops. Standard BFS finds the path with the fewest stops, but it doesn't care about the price. We need a hybrid!

Welcome to the **Bellman-Ford Algorithm**. Instead of a Min-Heap, this algorithm works in distinct "Waves" or "Layers".

**The Strategy:**
1. **The Price Board:** We create a board showing the cheapest known price to reach every airport. Initially, everything is `Infinity` except our starting airport (`src`), which is `0`.
2. **The Waves:** If we are allowed `k` stops, we can take at most `k + 1` flights. We will loop exactly `k + 1` times.
   * **Wave 1:** Find the cheapest prices using exactly 1 flight (0 stops).
   * **Wave 2:** Find the cheapest prices using up to 2 flights (1 stop).
   * **Wave 3:** Find the cheapest prices using up to 3 flights (2 stops).
3. **The Snapshot (Crucial Step):** Before every wave, we take a "snapshot" of the price board. When we calculate new prices, we only look at the snapshot. This guarantees we don't accidentally chain 5 flights together in a single wave!
4. **The Result:** After exactly `k + 1` waves, we look at the price board for our `dst`. If it is still `Infinity`, we couldn't reach it. Otherwise, that is our answer.

**The Math & Complexity Breakdown:**
* **Time Complexity:** $O(k \cdot E)$ — Where $E$ is the number of edges (flights). We loop through every single flight exactly $k + 1$ times.
* **Space Complexity:** $O(n)$ — We only need two arrays (the main array and the snapshot array) of size `n` to store the prices. We don't even need to build an Adjacency List!

In [ ]:
from typing import List

class Solution:
    def findCheapestPrice(self, n: int, flights: List[List[int]], src: int, dst: int, k: int) -> int:
        # Step 1: Initialize the price board
        # Every city costs Infinity to reach initially
        prices = [float('inf')] * n
        prices[src] = 0
        
        # Step 2: Run Bellman-Ford for exactly (k + 1) waves
        # If k = 1 stop, we can take at most 2 flights.
        for i in range(k + 1):
            
            # Take a snapshot of the prices at the start of this wave
            tmp_prices = prices.copy()
            
            # Loop through every single flight on the map
            for u, v, price in flights:
                
                # If the starting airport 'u' is unreachable in our snapshot, skip it.
                if prices[u] == float('inf'):
                    continue
                    
                # If (Cost to reach 'u' + Flight from 'u' to 'v') is cheaper than 
                # our current known price to 'v', update the temp array!
                if prices[u] + price < tmp_prices[v]:
                    tmp_prices[v] = prices[u] + price
                    
            # After checking all flights, make the snapshot the new official board
            prices = tmp_prices
            
        # Step 3: Check the final price for our destination
        if prices[dst] == float('inf'):
            return -1
        else:
            return prices[dst]